# 00 — Project Setup and FaceForensics++ Data Preparation

Run this notebook before the preprocessing and training notebooks.

It has two distinct responsibilities:

1. **Project setup:** create every required directory if it does not already exist.
2. **FF++ data preparation:** download the source videos, extract frames, and populate the full-frame `dataset/` splits used by the training notebooks.

Folder creation and normal reruns are non-destructive. Existing files are not deleted or overwritten.

## 1. Controls

`PREPARE_FFPP_DATA=True` performs the data-preparation work. Change it to `False` when you only want to mount Drive and create/check the folder structure.

This notebook deliberately has no automatic force-rebuild option. If it finds a partially populated dataset, it stops and asks you to inspect it rather than mixing two different splits.

In [12]:
INSTALL_DEPENDENCIES = True
PREPARE_FFPP_DATA = True

FRAMES_PER_VIDEO = 10
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
SEED = 42

assert FRAMES_PER_VIDEO > 0
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

print("Install dependencies:", INSTALL_DEPENDENCIES)
print("Prepare FF++ data:", PREPARE_FFPP_DATA)

Install dependencies: True
Prepare FF++ data: True


## 2. Mount Drive and import dependencies

In [13]:
from google.colab import drive
drive.mount("/content/drive")

if INSTALL_DEPENDENCIES:
    import subprocess
    import sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch", "torchvision", "opencv-python", "pandas",
        "scikit-learn", "kagglehub",
    ])

import json
import os
import random
import shutil
from pathlib import Path

import cv2
import kagglehub
import numpy as np
import pandas as pd
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda


## 3. Create the complete project directory structure

Every directory is created with `parents=True, exist_ok=True`. Therefore, an existing directory and everything inside it are preserved.

In [14]:
BASE_PATH = Path("/content/drive/MyDrive/deepfake_project")

EXTRACTED_FRAMES_DIR = BASE_PATH / "extracted_frames"
DATASET_DIR = BASE_PATH / "dataset"
FACE_DATASET_DIR = BASE_PATH / "face_dataset"
SBI_DATASET_DIR = BASE_PATH / "sbi_dataset"
MODEL_DIR = BASE_PATH / "saved_models"
RESULTS_DIR = BASE_PATH / "results"
CROSS_DATASET_RESULTS_DIR = RESULTS_DIR / "cross_dataset"
PLOTS_DIR = BASE_PATH / "plots"
CELEBDF_DIR = BASE_PATH / "celebdf_v2"
CELEBDF_FACES_DIR = CELEBDF_DIR / "pilot_faces"

PROJECT_DIRECTORIES = [
    BASE_PATH,
    EXTRACTED_FRAMES_DIR / "real",
    EXTRACTED_FRAMES_DIR / "fake",
    *[
        DATASET_DIR / split / class_name
        for split in ("train", "val", "test")
        for class_name in ("real", "fake")
    ],
    *[
        FACE_DATASET_DIR / split / class_name
        for split in ("train", "val", "test")
        for class_name in ("real", "fake")
    ],
    SBI_DATASET_DIR,
    CELEBDF_DIR / "pilot_frames" / "real",
    CELEBDF_DIR / "pilot_frames" / "fake",
    CELEBDF_FACES_DIR / "real",
    CELEBDF_FACES_DIR / "fake",
    MODEL_DIR,
    RESULTS_DIR,
    CROSS_DATASET_RESULTS_DIR,
    PLOTS_DIR,
]

directory_rows = []
for directory in PROJECT_DIRECTORIES:
    existed = directory.is_dir()
    directory.mkdir(parents=True, exist_ok=True)
    directory_rows.append({
        "directory": "." if directory == BASE_PATH else str(directory.relative_to(BASE_PATH)),
        "status": "already existed" if existed else "created",
    })

directory_status = pd.DataFrame(directory_rows)
display(directory_status)
print("Project root:", BASE_PATH)

,directory,status
0,.,created
1,extracted_frames/real,created
2,extracted_frames/fake,created
3,dataset/train/real,created
4,dataset/train/fake,created
5,dataset/val/real,created
6,dataset/val/fake,created
7,dataset/test/real,created
8,dataset/test/fake,created
9,face_dataset/train/real,created


Project root: /content/drive/MyDrive/deepfake_project


## Install the Fixed Experiment Manifests

The fixed FaceForensics++ split and Celeb-DF v2 pilot manifests are downloaded
from the project repository when they are not already present. Existing
manifest files are preserved.

In [15]:
from urllib.request import urlretrieve

MANIFEST_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "sawket/deepfake-detection-dissertation/main/manifests"
)

MANIFEST_DOWNLOADS = {
    BASE_PATH / "ffpp_video_split_manifest.json":
        f"{MANIFEST_BASE_URL}/ffpp_video_split_manifest.json",

    CELEBDF_DIR / "celebdf_pilot_manifest.json":
        f"{MANIFEST_BASE_URL}/celebdf_pilot_manifest.json",
}

for destination, source_url in MANIFEST_DOWNLOADS.items():
    if destination.exists():
        print("Manifest already exists:", destination)
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(source_url, destination)
        print("Downloaded manifest:", destination)

for manifest_path in MANIFEST_DOWNLOADS:
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest installation failed: {manifest_path}")

Downloaded manifest: /content/drive/MyDrive/deepfake_project/ffpp_video_split_manifest.json
Downloaded manifest: /content/drive/MyDrive/deepfake_project/celebdf_v2/celebdf_pilot_manifest.json


## 4. Download and inspect FaceForensics++

KaggleHub stores the downloaded source dataset in its cache. No download occurs when `PREPARE_FFPP_DATA=False`.

In [16]:
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv"}

if PREPARE_FFPP_DATA:
    download_path = Path(kagglehub.dataset_download("hungle3401/faceforensics"))
    ffpp_source = download_path / "FF++"
    source_dirs = {
        "real": ffpp_source / "real",
        "fake": ffpp_source / "fake",
    }

    for class_name, source_dir in source_dirs.items():
        if not source_dir.is_dir():
            raise FileNotFoundError(f"Missing FF++ source folder: {source_dir}")

    source_videos = {
        class_name: sorted(
            path for path in source_dir.iterdir()
            if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
        )
        for class_name, source_dir in source_dirs.items()
    }

    source_summary = pd.DataFrame([
        {"class": class_name, "videos": len(videos)}
        for class_name, videos in source_videos.items()
    ])
    display(source_summary)

    if any(len(videos) == 0 for videos in source_videos.values()):
        raise RuntimeError("No source videos were found for one or more FF++ classes.")
else:
    print("FF++ download and preparation skipped.")

100%|██████████| 2.73G/2.73G [02:26<00:00, 20.0MB/s]

Extracting files...


,class,videos
0,real,200
1,fake,200


## 5. Create or load a video-level split manifest

For a new preparation, videos—not individual frames—are assigned to train, validation, and test. This prevents frames from the same source video leaking across splits.

The manifest is saved and reused on subsequent runs, ensuring the split is reproducible.

In [17]:
SPLIT_MANIFEST_PATH = BASE_PATH / "ffpp_video_split_manifest.json"


def create_class_split(video_paths, seed):
    names = [path.name for path in video_paths]
    rng = random.Random(seed)
    rng.shuffle(names)

    total = len(names)
    train_end = int(TRAIN_RATIO * total)
    val_end = train_end + int(VAL_RATIO * total)
    return {
        "train": names[:train_end],
        "val": names[train_end:val_end],
        "test": names[val_end:],
    }


if PREPARE_FFPP_DATA:
    existing_split_dirs = [
        DATASET_DIR / split / class_name
        for split in ("train", "val", "test")
        for class_name in ("real", "fake")
    ]
    existing_split_counts = [
        sum(
            path.is_file()
            and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
            for path in folder.iterdir()
        )
        for folder in existing_split_dirs
    ]
    existing_splits_all_empty = all(count == 0 for count in existing_split_counts)

    if SPLIT_MANIFEST_PATH.exists():
        with SPLIT_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
            split_manifest = json.load(handle)
        print("Loaded existing split manifest:", SPLIT_MANIFEST_PATH)
    elif existing_splits_all_empty:
        split_manifest = {
            "seed": SEED,
            "split_unit": "source_video",
            "frames_per_video": FRAMES_PER_VIDEO,
            "ratios": {
                "train": TRAIN_RATIO,
                "val": VAL_RATIO,
                "test": TEST_RATIO,
            },
            "classes": {
                class_name: create_class_split(videos, SEED + index)
                for index, (class_name, videos) in enumerate(source_videos.items())
            },
        }
        with SPLIT_MANIFEST_PATH.open("w", encoding="utf-8") as handle:
            json.dump(split_manifest, handle, indent=2)
        print("Created split manifest:", SPLIT_MANIFEST_PATH)
    else:
        split_manifest = None
        print(
            "Existing dataset files were detected without a saved video-level manifest. "
            "They will be preserved and audited, but no new manifest will be attached to them."
        )

    if split_manifest is not None:
        manifest_rows = []
        for class_name, splits in split_manifest["classes"].items():
            for split, video_names in splits.items():
                manifest_rows.append({
                    "split": split,
                    "class": class_name,
                    "source_videos": len(video_names),
                    "expected_frames": len(video_names) * FRAMES_PER_VIDEO,
                })
        display(pd.DataFrame(manifest_rows).sort_values(["split", "class"]))

Loaded existing split manifest: /content/drive/MyDrive/deepfake_project/ffpp_video_split_manifest.json


,split,class,source_videos,expected_frames
5,test,fake,30,300
2,test,real,30,300
3,train,fake,140,1400
0,train,real,140,1400
4,val,fake,30,300
1,val,real,30,300


## 6. Extract missing frames

Ten deterministically spaced frames are extracted from each video. A video is skipped when all expected frame files already exist. Existing frame files are never overwritten.

In [18]:
def expected_frame_paths(video_path, output_dir):
    return [
        output_dir / f"{video_path.stem}_frame_{index}.jpg"
        for index in range(FRAMES_PER_VIDEO)
    ]


def extract_missing_frames(video_path, output_dir):
    output_paths = expected_frame_paths(video_path, output_dir)
    if all(path.exists() for path in output_paths):
        return 0, "already complete"

    capture = cv2.VideoCapture(str(video_path))
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        capture.release()
        return 0, "unreadable video"

    frame_indices = np.linspace(
        0, total_frames - 1, num=FRAMES_PER_VIDEO, dtype=int
    )
    written = 0
    for sample_index, frame_index in enumerate(frame_indices):
        output_path = output_paths[sample_index]
        if output_path.exists():
            continue
        capture.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        success, frame = capture.read()
        if success:
            cv2.imwrite(str(output_path), frame)
            written += 1
    capture.release()

    complete = all(path.exists() for path in output_paths)
    return written, "complete" if complete else "incomplete"


if PREPARE_FFPP_DATA:
    extraction_rows = []
    for class_name, videos in source_videos.items():
        output_dir = EXTRACTED_FRAMES_DIR / class_name
        for video_path in videos:
            written, status = extract_missing_frames(video_path, output_dir)
            extraction_rows.append({
                "class": class_name,
                "video": video_path.name,
                "new_frames": written,
                "status": status,
            })

    extraction_log = pd.DataFrame(extraction_rows)
    display(extraction_log.groupby(["class", "status"]).size().rename("videos"))

    incomplete = extraction_log[extraction_log["status"] == "incomplete"]
    if not incomplete.empty:
        raise RuntimeError(
            f"Frame extraction was incomplete for {len(incomplete)} video(s). "
            "Inspect extraction_log before continuing."
        )

,,videos
class,status,
fake,complete,200
real,complete,200


## 7. Populate the training dataset directories

The saved video manifest determines where each video's frames are copied. The notebook behaves as follows:

- **All six split folders empty:** populate them.
- **All six split folders already contain data:** preserve them and skip copying.
- **Only some folders contain data:** stop, because automatically mixing old and new splits would damage reproducibility.

In [19]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def image_count(folder):
    return sum(
        path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        for path in folder.iterdir()
    )


if PREPARE_FFPP_DATA:
    split_dirs = {
        (split, class_name): DATASET_DIR / split / class_name
        for split in ("train", "val", "test")
        for class_name in ("real", "fake")
    }
    counts_before = {
        key: image_count(folder) for key, folder in split_dirs.items()
    }

    all_empty = all(count == 0 for count in counts_before.values())
    all_populated = all(count > 0 for count in counts_before.values())

    if all_populated:
        print("All dataset split folders are already populated; copying skipped.")
    elif not all_empty:
        counts_table = pd.DataFrame([
            {"split": split, "class": class_name, "images": count}
            for (split, class_name), count in counts_before.items()
        ])
        display(counts_table)
        raise RuntimeError(
            "The dataset is only partially populated. No files were copied. "
            "Inspect the displayed counts and resolve the partial dataset manually."
        )
    else:
        copy_rows = []
        for class_name, splits in split_manifest["classes"].items():
            source_video_lookup = {
                path.name: path for path in source_videos[class_name]
            }
            for split, video_names in splits.items():
                destination = split_dirs[(split, class_name)]
                copied = 0
                for video_name in video_names:
                    video_path = source_video_lookup[video_name]
                    for source_frame in expected_frame_paths(
                        video_path, EXTRACTED_FRAMES_DIR / class_name
                    ):
                        if not source_frame.exists():
                            raise FileNotFoundError(
                                f"Expected extracted frame is missing: {source_frame}"
                            )
                        destination_frame = destination / source_frame.name
                        if not destination_frame.exists():
                            shutil.copy2(source_frame, destination_frame)
                            copied += 1
                copy_rows.append({
                    "split": split,
                    "class": class_name,
                    "copied_frames": copied,
                })
        display(pd.DataFrame(copy_rows).sort_values(["split", "class"]))
        print("Dataset folders populated successfully.")

,split,class,copied_frames
5,test,fake,300
2,test,real,300
3,train,fake,1400
0,train,real,1400
4,val,fake,300
1,val,real,300


Dataset folders populated successfully.


## 8. Final verification and leakage audit

This check confirms the image counts and verifies that no source-video filename appears in more than one split. Training should begin only after this section passes.

In [20]:
def recover_video_id(frame_filename):
    return frame_filename.rsplit("_frame_", 1)[0]


verification_rows = []
video_sets = {}
for split in ("train", "val", "test"):
    for class_name in ("real", "fake"):
        folder = DATASET_DIR / split / class_name
        image_files = [
            path.name for path in folder.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        ]
        video_ids = {recover_video_id(name) for name in image_files}
        video_sets[(split, class_name)] = video_ids
        verification_rows.append({
            "split": split,
            "class": class_name,
            "images": len(image_files),
            "source_videos": len(video_ids),
        })

verification_table = pd.DataFrame(verification_rows)
display(verification_table.sort_values(["split", "class"]))

overlap_rows = []
for class_name in ("real", "fake"):
    for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
        overlap = video_sets[(left, class_name)] & video_sets[(right, class_name)]
        overlap_rows.append({
            "class": class_name,
            "comparison": f"{left} vs {right}",
            "overlapping_videos": len(overlap),
        })

overlap_table = pd.DataFrame(overlap_rows)
display(overlap_table)

if (overlap_table["overlapping_videos"] > 0).any():
    print("WARNING: source-video overlap exists. Do not treat this as a clean video-level split.")
else:
    print("PASS: no source-video overlap was detected across train/val/test.")

if PREPARE_FFPP_DATA and (verification_table["images"] == 0).any():
    raise RuntimeError("At least one training split folder is empty.")

,split,class,images,source_videos
5,test,fake,300,30
4,test,real,300,30
1,train,fake,1400,140
0,train,real,1400,140
3,val,fake,300,30
2,val,real,300,30


,class,comparison,overlapping_videos
0,real,train vs val,0
1,real,train vs test,0
2,real,val vs test,0
3,fake,train vs val,0
4,fake,train vs test,0
5,fake,val vs test,0


PASS: no source-video overlap was detected across train/val/test.


## 9. Canonical checkpoint paths

These paths are shared by the modular training and evaluation notebooks.

In [ ]:
CHECKPOINTS = {
    "E1": MODEL_DIR / "baseline_cnn.pth",
    "E1.1": MODEL_DIR / "baseline_cnn_face.pth",
    "E2": MODEL_DIR / "efficientnet_b0.pth",
    "E2.1": MODEL_DIR / "efficientnet_b0_finetuned.pth",
    "E2.2": MODEL_DIR / "efficientnet_b0_face_finetuned.pth",
    "E3a": MODEL_DIR / "efficientnet_b0_sbi_best.pth",
    "E3b": MODEL_DIR / "efficientnet_b0_official_sbi_best.pth",
    "E3c": MODEL_DIR / "efficientnet_b0_official_sbi_sam_best.pth",
    "E3d": MODEL_DIR / "efficientnet_b4_official_sbi_best.pth",
    "E4": MODEL_DIR / "efficientnet_b0_fsbi_dwt_best.pth",
}

checkpoint_table = pd.DataFrame([
    {"experiment": experiment, "path": str(path), "exists": path.exists()}
    for experiment, path in CHECKPOINTS.items()
])
display(checkpoint_table)

,experiment,path,exists
0,E1,/content/drive/MyDrive/deepfake_project/saved_...,False
1,E1.1,/content/drive/MyDrive/deepfake_project/saved_...,False
2,E2,/content/drive/MyDrive/deepfake_project/saved_...,False
3,E2.1,/content/drive/MyDrive/deepfake_project/saved_...,False
4,E2.2,/content/drive/MyDrive/deepfake_project/saved_...,False
5,E3a,/content/drive/MyDrive/deepfake_project/saved_...,False
6,E3b,/content/drive/MyDrive/deepfake_project/saved_...,False
7,E3c,/content/drive/MyDrive/deepfake_project/saved_...,False
8,E3d,/content/drive/MyDrive/deepfake_project/saved_...,False
9,E4,/content/drive/MyDrive/deepfake_project/saved_...,False


## Setup and Preparation Complete

To reproduce the reported results using the supplied checkpoints, continue
with `02a_Face_Preprocessing.ipynb` and
`02c_B4_Test_Face_Preparation.ipynb`.

To retrain the models from the beginning, also run
`02b_SBI_Preprocessing.ipynb` and the required `03` training notebooks.

To run only the final E3b demonstrator, place the E3b checkpoint in
`saved_models/` and run
`optional/06b_E3b_Gradio_Application.ipynb`.